# IMMapp — LayoutXLM fine-tuning pe FATURA + RO sintetic (Kaggle)

Antrenează modelul real pe un GPU Kaggle (T4 x2 sau P100), folosind FATURA (engleză, 8.600 documente) combinat cu facturile românești sintetice (500 documente). Nu conține rezultate fabricate și nu modifică inferența locală până când arhiva modelului este importată explicit în IMMapp.

**Înainte de a rula**: în panoul din dreapta, **Notebook options → Accelerator**, selectează `GPU T4 x2` sau `GPU P100`. Activează Internet dacă e nevoie de instalarea pachetelor.

**2026-09-04**: modelul anterior antrenat cu acest notebook raporta test_token_accuracy 1.0 -- cifra era nereala, cauzata de scurgere de sablon (toate cele 43 de sabloane FATURA aparteau si in train, si in dev, si in test, deci testul cerea doar recunoasterea unui sablon deja vazut). Inainte de a impacheta arhiva `immapp-ro-dataset`, ruleaza local:

```bash
cd document-ai-backend/training
python3 split_by_template_holdout.py
```

Asta scrie `datasets/fatura/processed_holdout/` cu sabloane intregi tinute in afara antrenarii -- vezi comentariile din script si `kaggle_layoutxlm_training.md` sectiunea 0 pentru detalii. Foloseste `processed_holdout`, nu `processed_combined`, la pasul de arhivare de mai jos.

## 1. Atașează cele 3 input-uri Kaggle

Prin **+ Add Input**, atașează:

1. `immapp-project` — arhiva codului (`immate-dash-pro.zip`, fără `.venv`, `node_modules`, `dist`, `datasets/fatura/{raw,train,test,processed,processed_ro,processed_holdout}`).
2. `fatura-full` — `invoices_dataset_final.zip` (365MB), neschimbat.
3. `immapp-ro-dataset` — arhivă nouă cu structura:
   ```
   processed_holdout/layoutxlm_{train,dev,test}.jsonl
   processed_ro/images/*.png
   ```
   (labelurile combinate + imaginile RO; Kaggle extrage automat arhivele la upload).

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU CUDA indisponibil. Selectează un accelerator GPU."
print("CUDA disponibil:", torch.cuda.get_device_name(0), "| GPU count:", torch.cuda.device_count())

## 2. Localizează input-urile montate

`/kaggle/input` e read-only. Ajustează sloturile de mai jos dacă numele dataset-urilor diferă de cele folosite la upload.

In [ ]:
from pathlib import Path

PROJECT_ZIP = Path("/kaggle/input/immapp-project/immate-dash-pro.zip")
FATURA_ROOT = Path("/kaggle/input/fatura-full")
RO_DATASET_ROOT = Path("/kaggle/input/immapp-ro-dataset")

for candidate in (PROJECT_ZIP, FATURA_ROOT, RO_DATASET_ROOT):
    assert candidate.exists(), f"Lipsește: {candidate}. Verifică Add Input."
print("Input-uri găsite OK.")

## 3. Despachetează proiectul

In [ ]:
!unzip -q "{PROJECT_ZIP}" -d /kaggle/working
PROJECT_DIR = Path("/kaggle/working/immate-dash-pro")
%cd $PROJECT_DIR

## 4. Instalează dependențele

Instalarea Detectron2 poate dura câteva minute; mesajele de build sunt normale.

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r document-ai-backend/requirements-training.txt
!python -m pip install -q ninja cython pycocotools
!python -m pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!python -c "import torch, transformers, accelerate, detectron2; print('deps OK')"

## 5. Localizează căile din dataset-ul combinat

Fișierele `layoutxlm_{train,dev,test}.jsonl` din `processed_holdout` conțin căi absolute de pe MacBook. Scriptul de mai jos le rescrie ca să indice direct imaginile montate în Kaggle, fără citire din zip.

In [ ]:
fatura_images_dir = FATURA_ROOT / "invoices_dataset_final" / "images"
ro_images_dir = RO_DATASET_ROOT / "processed_ro" / "images"
ro_labels_dir = RO_DATASET_ROOT / "processed_holdout"

assert fatura_images_dir.exists(), fatura_images_dir
assert ro_images_dir.exists(), ro_images_dir
assert ro_labels_dir.exists(), ro_labels_dir

In [ ]:
!python document-ai-backend/training/localize_dataset_for_kaggle.py \
  --data "{ro_labels_dir}" \
  --fatura-images-dir "{fatura_images_dir}" \
  --ro-images-dir "{ro_images_dir}" \
  --output /kaggle/working/dataset_localized

## 6. Smoke-test pe GPU

Rulează doar două documente și un pas de optimizare. Greutățile temporare sunt șterse automat.

In [ ]:
!python document-ai-backend/training/train_layoutxlm_invoice_classifier.py \
  --data /kaggle/working/dataset_localized \
  --label-map document-ai-backend/datasets/fatura/inferred_label_map.json \
  --smoke-test --device cuda

## 7. Antrenare completă

Recomandare pentru cea mai bună calitate în limita unei sesiuni Kaggle: **6 epoci, batch 4, fp16**, cu selecția automată a celui mai bun checkpoint după acuratețea pe dev set (`load_best_model_at_end`, activat automat când există dev set). Pe P100/T4 cu 16GB VRAM ar trebui să încapă; dacă apare OOM, scade la `--batch-size 2`.

In [ ]:
!python document-ai-backend/training/train_layoutxlm_invoice_classifier.py \
  --data /kaggle/working/dataset_localized \
  --label-map document-ai-backend/datasets/fatura/inferred_label_map.json \
  --device cuda --fp16 --epochs 6 --batch-size 4 --save

# Variantă low-memory (rulează aceasta în locul comenzii de mai sus dacă apare OOM):
# !python document-ai-backend/training/train_layoutxlm_invoice_classifier.py \
#   --data /kaggle/working/dataset_localized \
#   --label-map document-ai-backend/datasets/fatura/inferred_label_map.json \
#   --device cuda --fp16 --epochs 6 --batch-size 2 --save

## 8. Arhivează modelul în Output

Rulează numai după ce antrenarea s-a terminat cu succes. Fișierele scrise în `/kaggle/working` apar automat în tab-ul **Output** al notebook-ului după **Save Version → Save & Run All**.

In [ ]:
import shutil

MODEL_DIR = PROJECT_DIR / "document-ai-backend/models/layoutxlm-invoice-token-classifier"
assert (MODEL_DIR / "config.json").exists(), "Modelul nu pare salvat corect."
archive_path = shutil.make_archive(
    "/kaggle/working/layoutxlm-invoice-token-classifier",
    "zip",
    root_dir=MODEL_DIR.parent,
    base_dir=MODEL_DIR.name,
)
print("Arhivă creată:", archive_path)

## 9. După descărcare

Descarcă `layoutxlm-invoice-token-classifier.zip` din secțiunea **Output** a notebook-ului, apoi pe MacBook:

```bash
cd /Users/mi/Desktop/immate-dash-pro
npm run document-ai:import-trained-model -- /path/to/layoutxlm-invoice-token-classifier.zip
npm run dev
curl http://localhost:8000/health
npm run document-ai:evaluate
```

Raportează numai metricile măsurate de `document-ai:evaluate`, nu rezultate presupuse.